In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://rdm.example.com/'
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
target_storage_id = 'nextcloud'
target_storage_name = 'Nextcloud'
webdav_host = None
webdav_username = None
webdav_password = None
webdav_folder = 'e2e'
rdm_project_name = 'TEST-WEBDAV-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))
target_file_view = 'file-tab'
skip_130mb_upload = False
skip_preview_check = True
close_on_fail = False
default_result_path = None
transition_timeout = 60000


In [ ]:
import tempfile
assert target_storage_id in ('owncloud', 'nextcloud')
if idp_username_1 is None:
    idp_username_1 = input(prompt='GRDM username: ')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt='GRDM password: ')
if webdav_host is None:
    webdav_host = input(prompt='WebDAV server URL: ')
if webdav_username is None:
    webdav_username = input(prompt='WebDAV username: ')
if webdav_password is None:
    webdav_password = getpass(prompt='WebDAV password: ')
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir


# GakuNinRDM 総合テスト [WebDAVストレージ-アカウント設定]

- サブシステム名: アドオン
- ページ/アドオン: ownCloud / Nextcloud
- 機能分類: ストレージ制御確認
- シナリオ名: 接続先選択・アカウント登録・フォルダ選択・ファイル操作
- 用意するテストデータ: 対象ストレージのアカウントを未登録のGRDMユーザー、接続可能URLリストに掲載されたWebDAV接続先、空の試験用フォルダ


In [ ]:
import importlib
import os
import scripts.playwright
importlib.reload(scripts.playwright)
from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること。

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await page.get_by_role('button', name='同意する', exact=True).click()
    await expect(page.get_by_role('button', name='同意する', exact=True)).to_have_count(0)

await run_pw(_step)


## ログイン情報を用いてGakuNin RDMにログインする

ダッシュボードが表示されること。

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## 指定された名前のプロジェクトを作成する

プロジェクト一覧に作成したプロジェクトが表示されること。

In [ ]:
async def _step(page):
    await grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)
    await expect(page.locator('[data-test-dashboard-item-title]').filter(has_text=rdm_project_name)).to_be_visible()

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること。

In [ ]:
async def _step(page):
    await page.locator('[data-test-dashboard-item-title]').filter(has_text=rdm_project_name).click()
    await expect(page.get_by_role('link', name='アドオン', exact=True)).to_be_visible()

await run_pw(_step)


## プロジェクト上部メニューの「アドオン」をクリックする

対象ストレージの「有効にする」が表示されること。

In [ ]:
async def _step(page):
    await page.get_by_role('link', name='アドオン', exact=True).click()
    await expect(page.locator(f'div[full_name="{target_storage_name}"]').get_by_text('有効にする', exact=True)).to_be_visible()

await run_pw(_step)


## 対象ストレージの「有効にする」をクリックする

確認ダイアログが表示されること。

In [ ]:
async def _step(page):
    await page.locator(f'div[full_name="{target_storage_name}"]').get_by_text('有効にする', exact=True).click()
    await expect(page.locator('button[data-bb-handler="confirm"]')).to_be_visible()

await run_pw(_step)


## 確認ダイアログの「確認」をクリックする

対象ストレージのアカウント接続リンクが表示されること。

In [ ]:
async def _step(page):
    await page.locator('button[data-bb-handler="confirm"]').click()
    await expect(page.locator(f'#{target_storage_id}Scope a[data-toggle="modal"]')).to_be_visible()

await run_pw(_step)


## 対象ストレージのアカウント接続リンクをクリックする

接続先選択と認証情報入力のダイアログが表示されること。

In [ ]:
async def _step(page):
    await page.locator(f'#{target_storage_id}Scope a[data-toggle="modal"]').click()
    await expect(page.locator('.modal.in #hostSelect')).to_be_visible()

await run_pw(_step)


## 接続先のドロップダウンから指定されたURLを選択する

指定されたURLが選択されること。

In [ ]:
async def _step(page):
    await page.locator('.modal.in #hostSelect').select_option(label=webdav_host)
    await expect(page.locator('.modal.in #hostSelect')).to_have_value(webdav_host)

await run_pw(_step)


## 対象ストレージのユーザー名とパスワードを入力する

認証情報が入力され、パスワードが伏せ字で表示されること。

In [ ]:
async def _step(page):
    await page.locator('.modal.in input[name="username"]').fill(webdav_username)
    await page.locator('.modal.in input[name="password"]').fill(webdav_password)
    await expect(page.locator('.modal.in input[name="password"]')).to_have_attribute('type', 'password')

await run_pw(_step)


## 接続ダイアログの「保存」をクリックする

ダイアログが閉じ、試験用フォルダが選択可能になること。

In [ ]:
async def _step(page):
    await page.locator('.modal.in').get_by_role('button', name='保存', exact=True).click()
    await expect(page.locator('.modal.in')).to_have_count(0)
    await expect(page.locator(f'//span[text()="{webdav_folder}"]/../../..//input[@type="radio"]')).to_be_enabled()

await run_pw(_step)


## 試験用フォルダのラジオボタンをクリックする

接続するフォルダ名が確認欄に表示されること。

In [ ]:
async def _step(page):
    await page.locator(f'//span[text()="{webdav_folder}"]/../../..//input[@type="radio"]').click()
    await expect(page.locator(f'.{target_storage_id}-confirm-selection')).to_be_visible()

await run_pw(_step)


## フォルダ選択の「保存」をクリックする

フォルダが正常にリンクされたメッセージが表示されること。

In [ ]:
async def _step(page):
    await page.locator(f'.{target_storage_id}-confirm-selection input[type="submit"]').click()
    await expect(page.locator(f'#{target_storage_id}Scope > .help-block').get_by_text('正常にリンクされました', exact=False)).to_be_visible()

await run_pw(_step)


## 「ファイルページ」リンクをクリックする

対象ストレージのファイル一覧が表示されること。

In [ ]:
async def _step(page):
    await page.get_by_role('link', name='ファイルページ', exact=True).click()
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible()

await run_pw(_step)


In [ ]:
await finish_pw_context()


## 対象ストレージでファイルの基本操作を行う

ストレージ共通のファイル基本操作の各期待値を満たすこと。

In [ ]:
from scripts.papermillHelpers import run_notebook

result_notebook = run_notebook(
    os.path.join(default_result_path, 'notebooks'),
    'テスト手順-ストレージ共通-ファイル基本操作.ipynb',
    transition_timeout,
    shared_params=dict(rdm_url=rdm_url, idp_name_1=idp_name_1,
                       idp_username_1=idp_username_1, idp_password_1=idp_password_1),
    extra_params=dict(target_storage_id=target_storage_id, target_storage_name=target_storage_name,
                      rdm_project_name=rdm_project_name, target_file_view=target_file_view,
                      enable_1gb_file_upload=False, skip_130mb_upload=skip_130mb_upload,
                      skip_preview_check=skip_preview_check),
)
result_notebook


# GakuNinRDM 総合テスト [WebDAVストレージ-アカウント切断]

- サブシステム名: アドオン
- ページ/アドオン: ownCloud / Nextcloud
- 機能分類: ストレージ制御確認
- シナリオ名: アカウント切断
- 用意するテストデータ: ファイル操作試験で登録したストレージアカウント


## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること。

In [ ]:
await init_pw_context(close_on_fail=close_on_fail, last_path=os.path.join(default_result_path, 'account-disconnection'))

async def _step(page):
    await page.goto(rdm_url)
    await page.get_by_role('button', name='同意する', exact=True).click()
    await expect(page.get_by_role('button', name='同意する', exact=True)).to_have_count(0)

await run_pw(_step)


## ログイン情報を用いてGakuNin RDMにログインする

ダッシュボードが表示されること。

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## ユーザーメニューから「設定」を選択する

ユーザー設定ページが表示されること。

In [ ]:
async def _step(page):
    await page.locator('div.nav-profile-name').click()
    await page.locator('[data-analytics-name="Settings"]').click()
    await expect(page.get_by_text('アドオンアカウント構成', exact=True)).to_be_visible()

await run_pw(_step)


## 左メニューの「アドオンアカウント構成」をクリックする

対象ストレージの登録済みアカウントが表示されること。

In [ ]:
async def _step(page):
    await page.get_by_text('アドオンアカウント構成', exact=True).click()
    await expect(page.locator(f'#{target_storage_id}AddonScope').get_by_text('アカウントを切断', exact=True)).to_have_count(1)

await run_pw(_step)


## 対象ストレージの「アカウントを切断」をクリックする

切断の確認ダイアログが表示されること。

In [ ]:
async def _step(page):
    await page.locator(f'#{target_storage_id}AddonScope').get_by_text('アカウントを切断', exact=True).click()
    await expect(page.locator('.bootbox-confirm .btn-danger')).to_be_visible()

await run_pw(_step)


## 確認ダイアログの「Disconnect」をクリックする

対象ストレージの登録済みアカウントがなくなること。

In [ ]:
async def _step(page):
    await page.locator('.bootbox-confirm .btn-danger').click()
    await expect(page.locator(f'#{target_storage_id}AddonScope').get_by_text('アカウントを切断', exact=True)).to_have_count(0)

await run_pw(_step)


終了処理を実施。

In [ ]:
await finish_pw_context()
